In [8]:
import torch
from torch.utils.data import DataLoader
from models import segFormer,STANet,swin_earlyfusion,BIT,Gswin_tac,UNet
from datasets.dataLoader import create_dataloaders
from loss.ce_dice_loss import CrossEntropyDiceLoss
from loss.boundary_loss import BoundaryLoss
from evaluation.metrics import compute_confusion_matrix, evaluate_all
from configs import train_config as config
from utils.forward_model import forward_model
from evaluation.evaluate import evaluate_model
from registry.model_registry import MODEL_REGISTRY
from logs.logger import Logger
logger = Logger(config.TrainConfig)


1. Training modular code

In [ ]:
import torch
from torch.cuda.amp import autocast, GradScaler
import os


def train_model(
    model,
    train_loader,
    val_loader,
    config,
    criterion,
    optimizer,
    model_type,
    scheduler,
    device,
    logger
):

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    scaler = GradScaler()

    best_miou = 0
    os.makedirs(config.checkpoint_dir, exist_ok=True)

    for epoch in range(config.num_epochs):

        print(f"\n===== Epoch {epoch+1}/{config.num_epochs} =====")

        # ===============================
        # TRAIN
        # ===============================

        model.train()
        train_loss = 0
        num_batches = 0

        for t1, t2, mask, file_id in train_loader:

            t1 = t1.to(device)
            t2 = t2.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            with autocast():
                pred = forward_model(model, t1, t2, model_type)
                loss = criterion(pred, mask)

            # 🔥 CHECK LOSS
            if not torch.isfinite(loss):
                print("Invalid loss detected in:", file_id)
                print("T1 min/max:", t1.min().item(), t1.max().item())
                print("T2 min/max:", t2.min().item(), t2.max().item())
                print("Mask unique:", torch.unique(mask))
                raise RuntimeError(f"Invalid loss at {file_id}")

            # 🔥 CHECK PREDICTION
            if not torch.isfinite(pred).all():
                print("Invalid prediction in:", file_id)
                raise RuntimeError(f"Invalid prediction at {file_id}")

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            num_batches += 1

        train_loss /= num_batches
        print(f"Train Loss: {train_loss:.4f}")

        # ===============================
        # VALIDATION
        # ===============================

        model.eval()
        val_loss = 0
        num_val_batches = 0

        conf_matrix = torch.zeros(
            (config.num_classes, config.num_classes)
        ).to(device)

        with torch.no_grad():

            for t1, t2, mask, file_id in val_loader:

                t1 = t1.to(device)
                t2 = t2.to(device)
                mask = mask.to(device)

                with autocast():
                    pred = forward_model(model, t1, t2, model_type)
                    loss = criterion(pred, mask)

                # 🔥 VALIDATION CHECK
                if not torch.isfinite(loss):
                    print("VAL ERROR:", file_id)
                    raise RuntimeError("Invalid validation loss")

                if not torch.isfinite(pred).all():
                    print("Invalid prediction in:", file_id)
                    raise RuntimeError("Invalid prediction")

                val_loss += loss.item()
                num_val_batches += 1

                pred_label = torch.argmax(pred, dim=1)

                conf_matrix += compute_confusion_matrix(
                    pred_label,
                    mask,
                    config.num_classes
                )

        val_loss /= num_val_batches

        results = evaluate_all(conf_matrix)
        miou = results["Mean IoU"]

        # ===============================
        # LOGGER
        # ===============================

        metrics = {
            "iou": results.get("Mean IoU", 0),
            "f1": results.get("Mean F1", 0),
            "precision": results.get("Precision", 0),
            "recall": results.get("Recall", 0),
            "accuracy": results.get("Accuracy", 0),
            "kappa": results.get("Kappa", 0)
        }

        if logger:
            logger.log(
                epoch=epoch + 1,
                train_loss=train_loss,
                val_loss=val_loss,
                metrics=metrics
            )

            logger.print(
                epoch=epoch + 1,
                train_loss=train_loss,
                val_loss=val_loss,
                metrics=metrics
            )

        # ===============================
        # SCHEDULER
        # ===============================

        if scheduler:
            try:
                scheduler.step(val_loss)  # untuk ReduceLROnPlateau
            except:
                scheduler.step()  # untuk StepLR, dll

        # ===============================
        # SAVE BEST MODEL
        # ===============================

        if miou > best_miou:

            best_miou = miou

            if config.save_model:

                path = os.path.join(
                    config.checkpoint_dir,
                    f"{config.model_name}_best.pth"
                )

                torch.save(model.state_dict(), path)
                print("Best model saved!")

    print("Training finished.")


2. Init DataLoader

In [17]:
root_dir = "data/"

train_loader, val_loader, test_loader = create_dataloaders(
    root_dir=root_dir,
    batch_size=config.TrainConfig.batch_size
)
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

2
1
1


3. Init parameters

In [18]:
criterion_model = CrossEntropyDiceLoss()
criterion_gswintac = BoundaryLoss()

model_info = MODEL_REGISTRY["unet_ef"]
model = model_info["model"]()
model_type = model_info["type"]
optimizer_name = model_info["optimizer"]

if optimizer_name == "Adam":

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.TrainConfig.learning_rate
    )

elif optimizer_name == "AdamW":

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.TrainConfig.learning_rate,
        weight_decay=config.TrainConfig.weight_decay
    )

scheduler = None

if config.TrainConfig.scheduler == "cosine":

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.TrainConfig.t_max
    )

4. Train the models ( makesure to change model_name in config first )

In [22]:
if(config.TrainConfig.model_name != "gswin_tac") :criterion = criterion_model
else: criterion = criterion_gswintac

train_model(
    model= model,
    train_loader=train_loader,
    val_loader = val_loader,
    config=config.TrainConfig,
    criterion=criterion,
    optimizer= optimizer,
    model_type= model_type,
    scheduler= scheduler,
    device="cpu"
)


===== Epoch 1/30 =====


/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_4887/2141387299.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_4887/2141387299.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Train Loss: 1.7794


/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_4887/2141387299.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Epoch 1
Train Loss : 1.7794
Val Loss   : 1.7656
IOU       : 0.1114
F1        : 0.1670
PRECISION : 0.0000
RECALL    : 0.0000
ACCURACY  : 0.0000
KAPPA     : 0.0000
Best model saved!

===== Epoch 2/30 =====
Train Loss: 1.7656

Epoch 2
Train Loss : 1.7656
Val Loss   : 1.7656
IOU       : 0.1114
F1        : 0.1670
PRECISION : 0.0000
RECALL    : 0.0000
ACCURACY  : 0.0000
KAPPA     : 0.0000
Best model saved!

===== Epoch 3/30 =====
Train Loss: 1.7532

Epoch 3
Train Loss : 1.7532
Val Loss   : 1.7656
IOU       : 0.1115
F1        : 0.1673
PRECISION : 0.0000
RECALL    : 0.0000
ACCURACY  : 0.0000
KAPPA     : 0.0000
Best model saved!

===== Epoch 4/30 =====
Train Loss: 1.7418

Epoch 4
Train Loss : 1.7418
Val Loss   : 1.7657
IOU       : 0.1135
F1        : 0.1713
PRECISION : 0.0000
RECALL    : 0.0000
ACCURACY  : 0.0000
KAPPA     : 0.0005
Best model saved!

===== Epoch 5/30 =====
Train Loss: 1.7312

Epoch 5
Train Loss : 1.7312
Val Loss   : 1.7659
IOU       : 0.1197
F1        : 0.1841
PRECISION : 0.000

5. TEST the models 

In [ ]:
model_path = os.path.join(
    config.TrainConfig.checkpoint_dir,
    f"{config.TrainConfig.experiment_name}_best.pth"
)

evaluate_model(
    model=model,
    loader=test_loader,
    model_path=model_path,
    num_classes=config.TrainConfig.num_classes,
    model_type=model_type,
    batch_size=config.TrainConfig.batch_size,
    device=config.TrainConfig.device
)